# ECHO-17: preserved real-data TabPFN experiment

Inference is unverified until this notebook completes. In a **fresh GPU runtime**, upload the locally exported `echo17_mission3_ready.zip`. This notebook preserves the existing measured features, ordered split and RF results; it does not download UCI or retrain the baseline.

Select **Runtime > Change runtime type > GPU**. Complete Prior Labs authentication and accept the separate [non-commercial checkpoint terms](https://huggingface.co/Prior-Labs/tabpfn_3_5) if requested. Keep credentials out of cells and outputs. See the bundled `docs/TABPFN_HANDOFF.md` for local export/import commands.

UCI credit: Helwig, Pignanelli and Schuetze (2015), DOI 10.24432/C5CW21, CC BY 4.0. Six measured channels, 36 predictors, 1,764 TRAIN / 441 TEST cycles.


In [ ]:
import hashlib
import io
import json
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from google.colab import files


def unpack_bundle(uploaded, destination):
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one exported input ZIP")
    destination = Path(destination)
    if destination.exists():
        raise ValueError("Destination exists; use a fresh runtime to preserve prior work")
    with ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        manifest_name = "echo-17/colab_manifest.json"
        manifest = json.loads(archive.read(manifest_name))
        names = archive.namelist()
        if manifest["schema_version"] != 1 or set(names) != set(manifest["files"]) | {manifest_name} or len(names) != len(set(names)):
            raise ValueError("Invalid or incomplete handoff manifest")
        contents = {}
        for name, digest in manifest["files"].items():
            path = PurePosixPath(name)
            if path.is_absolute() or path.parts[0] != "echo-17" or ".." in path.parts or "\\" in name or ":" in name:
                raise ValueError("Unsafe bundle path")
            value = archive.read(name)
            if hashlib.sha256(value).hexdigest() != digest:
                raise ValueError(f"Checksum mismatch: {name}")
            contents[name] = value
        for name, value in contents.items():
            target = destination.joinpath(*PurePosixPath(name).parts[1:])
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(value)
        (destination / "colab_manifest.json").write_text(json.dumps(manifest, indent=2))
    print(f"Verified and unpacked {len(contents)} files; no inference run yet")
    return destination / "backend"


backend = unpack_bundle(files.upload(), "/content/echo-17")


In [ ]:
import subprocess
import sys

# check=True stops on errors. No shell commands silently continue after failure.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn", "pandas>=2.1,<3", "scikit-learn>=1.5,<2"], cwd=backend, check=True)
subprocess.run([sys.executable, "-m", "ml.train_tabpfn", "--check-inputs"], cwd=backend, check=True)


## Genuine inference and model access

The next cell explicitly selects `ModelVersion.V3_5` and seed 42. It may open an authentication flow or print an authorization link. Complete the private model-access flow when requested. An access failure or missing GPU is a blocker, not a successful result. Do not replace it with example predictions.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Select a GPU runtime; keep the RF demo if GPU access remains blocked.")
from importlib.metadata import version
from tabpfn.constants import ModelVersion
print("GPU:", torch.cuda.get_device_name(0), "CUDA:", torch.version.cuda)
print("TabPFN package:", version("tabpfn"), "seed:", 42)
print("Official requested models:", ModelVersion.V3_5, ModelVersion.V3_5_FAST)
if input("Review https://huggingface.co/Prior-Labs/tabpfn_3_5 and its weight license. Type I ACCEPT only if you accept the terms for this research evaluation: ").strip() != "I ACCEPT":
    raise RuntimeError("License acceptance required before model access. No inference performed.")
# Authentication, if prompted, must use the private official flow; never paste tokens into cells.
subprocess.run([sys.executable, "-m", "ml.train_tabpfn", "--device", "cuda", "--allow-fast-fallback"], cwd=backend, check=True)


In [ ]:
# Read only actual output files produced by the successful preceding run.
for name in ("baseline_metrics.json", "tabpfn_metrics.json"):
    metric = json.loads((backend / "artifacts" / name).read_text())
    print(name, {key: metric[key] for key in ("model", "accuracy", "macro_f1", "train_count", "test_count", "wrong_cycle_ids")})
subprocess.run([sys.executable, "-m", "ml.comparison"], cwd=backend, check=True)
comparison = json.loads((backend / "artifacts/model_comparison.json").read_text())
print("Class-wise F1:", {model: {label: comparison[model]["classification_report"][label]["f1-score"] for label in ("3", "20", "100")} for model in ("baseline", "tabpfn")})
print("All disagreement / improvement / regression cases:", comparison["cases"])


In [ ]:
# Recompute scores and check provenance before exporting only TabPFN outputs.
result_zip = backend / "artifacts" / "echo17_results.zip"
subprocess.run([sys.executable, "-m", "ml.colab_bundle", "results", str(result_zip)], cwd=backend, check=True)
files.download(str(result_zip))


## Return to the local app

Save the downloaded ZIP as `backend/artifacts/echo17_results.zip`. From local `backend`, run:

```powershell
& ./.venv/mission1/Scripts/python.exe -m ml.colab_bundle check-results artifacts/echo17_results.zip
& ./.venv/mission1/Scripts/python.exe -m ml.colab_bundle import artifacts/echo17_results.zip
& ./.venv/mission1/Scripts/python.exe -m ml.comparison
& ./.venv/mission1/Scripts/python.exe -m pytest -q
```

The updated backend revalidates TabPFN files on access. Restart it if running older code, then verify the diagnosis model selector and evaluation comparison. Record genuine results and failure cases in `docs/MODEL_RESULTS.md`. Predictions served by the demo remain **precomputed from model inference**. Do not mark M01-M03 complete from local input/bundle checks alone.
